# Prompt variants for the LLM interest rating

**Purpose.** Compare the main prompt in `llm_rate.py` with several other wordings of the same question.
Every variant is run on the same local model, read out the same way, and scored with the same cross-validation,
so the only difference is the prompt.

- **Model:** `mlx-community/Qwen2.5-VL-7B-Instruct-4bit` (Apache-2.0), run locally with MLX, text only.
- **Readout:** one forward pass per title; the next-token probabilities of "1"…"6" are rescaled to sum to 1 and
  turned into the expected rating `1·p1 + … + 6·p6`. Zero-shot: no label is ever shown to the model.
- **Evaluation:** ridge (α = 1, standardised inputs) on [rating + 4 source one-hots], RepeatedKFold
  5 folds × 10 repeats, seed 42, MAE against the 120 single-rater train labels (same as `llm.ipynb`).
- **Cache:** scores are saved to `features/prompt_variants.csv`; a rerun only scores variants that are missing.
  Existing scripts and feature files are not changed.

In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold

SEED = 42
MODEL = "mlx-community/Qwen2.5-VL-7B-Instruct-4bit"
CACHE = Path("features/prompt_variants.csv")

train = pd.read_csv("data/train.csv"); test = pd.read_csv("data/test.csv")
df = pd.concat([train, test], ignore_index=True)
KIND = {"youtube": "YouTube video", "reddit": "Reddit post", "podcast": "podcast episode", "article": "news article"}
SCALE = "Scale: 1 = not interesting at all, 2 = slightly, 3 = somewhat, 4 = fairly, 5 = very, 6 = extremely interesting."
MAIN_SYSTEM = "You are an average adult internet user judging content titles. Answer only with what is asked."
print(len(df), "titles")

150 titles


## 1. Prompt variants

Each variant changes one part of the main prompt. `reverse=True` means a high answer is a *low* interest, so the
expected value is flipped (`7 − value`) before evaluation.

| Name | What changes |
|---|---|
| `main` | The prompt in `llm_rate.py` (reference) |
| `no_system` | No system message (no "average adult internet user" role) |
| `no_source` | The source type (YouTube video, Reddit post, …) is left out |
| `no_scale_labels` | Only "1 to 6", without the words for each level |
| `personal` | "How interesting is this title **to you**?" instead of "to a typical reader" |
| `editor` | The model judges as a professional editor rating title **quality** |
| `boring` | The question is reversed: "How **boring** …?" |

In [2]:
VARIANTS = {
    "main": dict(system=MAIN_SYSTEM, user=(
        "How interesting would a typical reader find this {kind} title?\n" + SCALE +
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "no_system": dict(system=None, user=(
        "How interesting would a typical reader find this {kind} title?\n" + SCALE +
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "no_source": dict(system=MAIN_SYSTEM, user=(
        "How interesting would a typical reader find this title?\n" + SCALE +
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "no_scale_labels": dict(system=MAIN_SYSTEM, user=(
        "How interesting would a typical reader find this {kind} title, on a scale from 1 to 6?"
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "personal": dict(system=MAIN_SYSTEM, user=(
        "How interesting is this {kind} title to you?\n" + SCALE +
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "editor": dict(system="You are a professional editor who reviews headlines.", user=(
        "Rate the quality of this {kind} title from 1 (very poor) to 6 (excellent)."
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
    "boring": dict(system=MAIN_SYSTEM, reverse=True, user=(
        "How boring would a typical reader find this {kind} title?\n"
        "Scale: 1 = not boring at all, 2 = slightly, 3 = somewhat, 4 = fairly, 5 = very, 6 = extremely boring."
        "\nTitle: {title}\nAnswer with a single digit from 1 to 6.")),
}
print("example (first title, main):\n" + VARIANTS["main"]["user"].format(kind=KIND[df.domain[0]], title=df.title[0]))

example (first title, main):
How interesting would a typical reader find this YouTube video title?
Scale: 1 = not interesting at all, 2 = slightly, 3 = somewhat, 4 = fairly, 5 = very, 6 = extremely interesting.
Title: YFG Fatso - "Bougie Hoes Pt 2 [Official Music Video]
Answer with a single digit from 1 to 6.


## 2. Scoring with the local LLM

Same readout as `llm_rate.py`, including the reset of the cached RoPE positions in mlx-vlm before every call
(without it, every title after the first gets wrong probabilities). Takes about 1–2 minutes per variant.

In [3]:
cache = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame({"title_id": df.title_id})
todo = [v for v in VARIANTS if f"{v}_exp" not in cache]
print("already scored:", [v for v in VARIANTS if v not in todo], "| to score:", todo)

if todo:
    import mlx.core as mx
    from mlx_vlm import load
    model, proc = load(MODEL)
    tok, lm = proc.tokenizer, model.language_model
    digit_ids = [tok.encode(str(d), add_special_tokens=False)[0] for d in range(1, 7)]

    def digit_probs(system, user):
        """Probabilities of "1".."6" as the first answer token (not rescaled)."""
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        lm._rope_deltas = None; lm._position_ids = None          # mlx-vlm RoPE cache reset
        out = lm(mx.array([tok.encode(s)]))
        logits = getattr(out, "logits", out)[0, -1].astype(mx.float32)
        return np.array(mx.softmax(logits))[digit_ids]

    for v in todo:
        cfg, t0, exp, mass = VARIANTS[v], time.time(), [], []
        for r in df.itertuples():
            p = digit_probs(cfg["system"], cfg["user"].format(kind=KIND[r.domain], title=r.title))
            mass.append(p.sum()); p = p / p.sum()
            e = float((p * np.arange(1, 7)).sum())
            exp.append(7 - e if cfg.get("reverse") else e)
        cache[f"{v}_exp"], cache[f"{v}_mass"] = exp, mass
        cache.to_csv(CACHE, index=False)                          # save after each variant
        print(f"{v:16s} {time.time() - t0:5.0f}s  mean digit mass {np.mean(mass):.3f}")
    json.dump({"model": MODEL, "variants": VARIANTS}, open("features/prompt_variants_prompts.json", "w"), indent=2)

already scored: [] | to score: ['main', 'no_system', 'no_source', 'no_scale_labels', 'personal', 'editor', 'boring']


/Users/home/workspace/School/ML/competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1122 18:22:49.180000 99977 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 3409.08it/s]

main               100s  mean digit mass 1.000


... (2 outputs left out)

no_scale_labels     62s  mean digit mass 1.000


personal            77s  mean digit mass 1.000


editor              69s  mean digit mass 1.000


boring              84s  mean digit mass 1.000


## 3. Cross-validation

Same ridge and splits as `llm.ipynb`. Columns:
- `cv_mae`: mean MAE over 50 folds (lower is better; differences below ~0.02 are noise);
- `spearman`: rank correlation of the raw rating with the train labels (higher is better);
- `corr_main`: correlation with the main prompt's rating (how different the variant really is);
- `digit_mass`: share of probability on "1"…"6" (close to 1 = the model followed the format).

In [4]:
d = df.merge(cache, on="title_id")
tr = d.iloc[:len(train)]
y = tr.interest.values.astype(float)
dom = pd.get_dummies(d.domain).reindex(columns=["article", "podcast", "reddit", "youtube"], fill_value=0).values.astype(float)
cv = list(RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED).split(y))

def cv_mae(score):
    X = np.column_stack([score, dom])[:len(train)]
    maes = []
    for a, b in cv:
        m = make_pipeline(StandardScaler(), Ridge(alpha=1)).fit(X[a], y[a])
        maes.append(np.abs(np.clip(m.predict(X[b]), 1, 6) - y[b]).mean())
    return np.mean(maes), np.std(maes)

rows = []
for v in VARIANTS:
    s = d[f"{v}_exp"].values
    mae, sd = cv_mae(s)
    rows.append(dict(variant=v, cv_mae=mae, fold_sd=sd, spearman=spearmanr(s[:len(train)], y)[0],
                     corr_main=np.corrcoef(s, d.main_exp)[0, 1], digit_mass=d[f"{v}_mass"].mean()))
res = pd.DataFrame(rows).set_index("variant").sort_values("cv_mae")
print("reference: constant mean 1.495")
res.round(3)

reference: constant mean 1.495


,cv_mae,fold_sd,spearman,corr_main,digit_mass
variant,,,,,
editor,1.399,0.148,0.302,0.738,1.0
no_scale_labels,1.410,0.181,0.214,0.947,1.0
main,1.415,0.177,0.190,1.000,1.0
no_source,1.431,0.172,0.208,0.941,1.0
personal,1.431,0.176,0.177,0.974,1.0
no_system,1.433,0.174,0.150,0.964,1.0
boring,1.447,0.166,0.210,0.363,1.0


In [5]:
# check: the 'main' variant should reproduce the score in features/llm_scores.csv (same prompt, same model)
ref = pd.read_csv("features/llm_scores.csv")[["title_id", "interest_exp"]].merge(d[["title_id", "main_exp"]])
print("max |difference| vs llm_scores.csv:", round(float((ref.interest_exp - ref.main_exp).abs().max()), 4))

max |difference| vs llm_scores.csv: 0.0


## 4. Per-source Spearman

The rating should help inside every source, not only through differences between sources.

In [6]:
per = {v: {g: spearmanr(grp[f"{v}_exp"], grp.interest)[0] for g, grp in tr.groupby("domain")} for v in VARIANTS}
pd.DataFrame(per).T.loc[res.index].round(3)

,article,podcast,reddit,youtube
editor,0.215,0.522,0.285,0.266
no_scale_labels,-0.120,0.401,0.375,0.255
main,-0.059,0.357,0.337,0.231
no_source,-0.119,0.418,0.278,0.217
personal,-0.128,0.392,0.285,0.263
no_system,-0.047,0.338,0.317,0.215
boring,0.137,0.242,0.298,0.146


## 5. Summary

- The `main` prompt reproduces `features/llm_scores.csv` exactly (max difference 0.0), so the comparison is fair.
- Every variant followed the answer format (digit mass 1.000).
- Worse than `main` (1.415): `no_system` 1.433, `no_source` 1.431, `personal` 1.431, `boring` 1.447. Dropping the
  "average reader" role, the source type or the "typical reader" wording each hurt, and the reversed "boring"
  question gives a quite different ranking (correlation 0.36 with `main`).
- `no_scale_labels` (1.410) is a tie with `main` (difference below the ~0.02 noise level; correlation 0.95).
- `editor` (title **quality** judged by an editor) scored better: CV 1.399, Spearman 0.30 vs 0.19, and positive on
  news articles, where `main` is slightly negative. This variant was found after the submissions were made and
  was not used for any submission.